# FTS Custom ML Backtesting Workspace

Welcome to the custom backtesting workspace! This notebook demonstrates how to run a realistic simulation of a machine learning-based trading strategy using the Financial Trading System (FTS) framework.

### Features of this Backtest:
1. **Historical Replay Event Loop:** Ticks are played back chronologically from our SQLite database.
2. **Forecasting Strategy:** Uses a pre-trained model to predict price direction from historical close prices.
3. **Execution Delay:** Simulates a realistic **execution delay** using `KBarExecuteDelay()` (meaning a signal generated at tick $T$ is submitted, and execution/fill status is checked at tick $T+k$).
4. **Price Slippage:** Simulates **price slippage** on both buy and sell orders using `FlatPriceSlip`.
5. **Visual Inspection:** Uses the `BacktestVisualizer` to display predictions overlaid with buy/sell trade markers (derived cleanly from the filled `order_logs`).
6. **Performance Summary Metrics:** Calculates and exports annualized returns, Sharpe ratio, and maximum drawdown metrics.

### 1. Import Dependencies

In [1]:
import os
import json
import logging
import pandas as pd
import numpy as np
from datetime import datetime, timezone
from sqlalchemy import create_engine

# Core FTS components & Backtest Specification
from trading_bot.config import settings
from trading_bot.core.database import init_db, SessionLocal
from trading_bot.core.loop import HistoricalReplayLoop
from trading_bot.core.pipeline import TradingPipeline
from trading_bot.monitoring.prediction_logger import DatabasePredictionLogger
from trading_bot.core.models import BacktestPredictionLog, ModelRegistryLog, OrderLog as OrderLogModel, TradeLog as TradeLogModel, Position as PositionModel
from trading_bot.core.repository import MarketDataRepository, ModelRepository, OrderRepository, PositionRepository
from trading_bot.core.schemas import BarData, OrderSide, OrderStatus
from trading_bot.backtesting import BacktestSpec

# ML/Strategy and Risk components
from nets.output_selectors import DynamicThresholdClassifier
from nets.inference import ONNXPredictor
from nets.strategies.nets_strategy import NetsStrategy
from trading_bot.core.transforms import LogReturnTransform
from trading_bot.strategy.engine import StrategyEngine
from trading_bot.risk_management.portfolio import Portfolio
from trading_bot.risk_management.sizing.fixed_percentage import FixedPercentageSizer
from trading_bot.risk_management.manager import RiskManager

# Execution & Backtest components
from trading_bot.execution.delay import KBarExecuteDelay
from trading_bot.execution.slippage import FlatPriceSlip
from trading_bot.execution.handlers.simulated_handler import SimulatedExecutionHandler
from trading_bot.execution.engine import ExecutionEngine
from trading_bot.backtesting.readers import SQLBacktestDataReader
from trading_bot.backtesting import BacktestVisualizer, HTMLBacktestExporter

# Set logging level to INFO for detailed simulation traces
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger()

CHOSEN_MARKET = "BTCUSDT"
CHOSEN_MODEL = "xgboost"
CHOSEN_FEATURES = "ohlcv"

# Load canonical Backtest Specification
spec_path = f'./specs/backtests/{CHOSEN_MARKET}/{CHOSEN_MODEL}_{CHOSEN_FEATURES}_backtest.yaml'
spec = BacktestSpec.from_yaml(spec_path)
print(f"Successfully loaded canonical BacktestSpec from: {spec_path}")

Successfully loaded canonical BacktestSpec from: ./specs/backtests/BTCUSDT/xgboost_ohlcv_backtest.yaml


### 2. Setup SQLite Database & Connect

We connect to the local SQLite database and clear any existing logs associated with our specific `run_id` to ensure a clean backtest run, without dropping the tables.

In [2]:

db = SessionLocal()

run_id = spec.run_id

# Clear logs from previous runs of this specific backtest to ensure clean metrics
db.query(BacktestPredictionLog).filter_by(run_id=run_id).delete()
db.query(OrderLogModel).filter_by(run_id=run_id).delete()
db.query(TradeLogModel).filter_by(run_id=run_id).delete()
db.query(PositionModel).filter_by(run_id=run_id).delete()
db.commit()

print("Connected to database and cleared logs for run ID:", run_id)

Connected to database and cleared logs for run ID: presentation_xgboost_ohlcv


### 3. Setup Strategy and Prediction Logic

We load our pre-trained model, set up feature transformation, and configure a threshold classifier output selector.

In [3]:
with SessionLocal() as db_session:
    model_repo = ModelRepository(db_session)
    feature_cols = spec.features.feature_cols
    
    # 1. Look for active production model matching signature and feature set
    model_entry = model_repo.get_production_model(
        model_type=spec.model_type,
        instrument_id=spec.market.instrument_id,
        interval=spec.market.interval,
        horizon=spec.horizon,
        feature_cols=feature_cols
    )
    if model_entry is not None:
        onnx_path = model_entry.onnx_path
        print(f"Loaded production model '{model_entry.model_id}' from registry (features={feature_cols}): {onnx_path}")
    else:
        # 2. Fall back to latest candidate model matching signature and feature set
        model_entry = model_repo.get_candidate_model(
            model_type=spec.model_type,
            instrument_id=spec.market.instrument_id,
            interval=spec.market.interval,
            horizon=spec.horizon,
            feature_cols=feature_cols
        )
        if model_entry is not None:
            onnx_path = model_entry.onnx_path
            print(f"No production model found. Falling back to candidate '{model_entry.model_id}' from registry (features={feature_cols}): {onnx_path}")
        else:
            fallback_onnx = f'../models/my_{spec.model_type}_model.onnx'
            if os.path.exists(fallback_onnx):
                onnx_path = fallback_onnx
                print(f"No model found in registry matching signature (features={feature_cols}). Falling back to local file: {onnx_path}")
            else:
                raise ValueError(
                    f"No model in registry or filesystem matches requested configuration: "
                    f"model_type='{spec.model_type}', instrument_id='{spec.market.instrument_id}', "
                    f"interval='{spec.market.interval}', horizon={spec.horizon}, feature_cols={feature_cols}"
                )

predictor = ONNXPredictor(onnx_path)
output_selector = DynamicThresholdClassifier(
    k=spec.classifier.classifier_k, 
    period=spec.classifier.period, 
    confidence_multiplier=spec.classifier.confidence_multiplier
)

strategy = NetsStrategy(
    predictor=predictor,
    output_selector=output_selector,
    lookback_period=predictor.model_metadata.lookback_period if predictor.model_metadata else spec.features.lookback_period,
    name_suffix=spec.model_type,
    feature_cols=feature_cols,
    allow_in_sample=spec.classifier.allow_in_sample
)
strategy_engine = StrategyEngine(strategies=[strategy])

ValueError: No model in registry or filesystem matches requested configuration: model_type='xgboost', instrument_id='BTC/USDT', interval='30m', horizon=1, feature_cols=['open', 'high', 'low', 'close', 'volume']

### 4. Build Backtesting Pipeline with Delay and Slippage Models

Here we instantiate the components required for a realistic backtest simulation:
- **Execution Delay:** `KBarExecuteDelay(k=1)` is injected into the simulated handler.
- **Slippage:** `FlatPriceSlip(slippage_pct=0.001)` (0.1% price penalty) is injected into the simulated handler.
- **Portfolio & Sizer:** A standard portfolio initialized with $10,000, sizing positions at 10% of total equity per trade.

In [ ]:
pos_repo = PositionRepository(db)
order_repo = OrderRepository(db)

portfolio = Portfolio(
    initial_balance=spec.execution.initial_balance,
    quote_currency=spec.execution.quote_currency,
    pos_repo=pos_repo,
    order_repo=order_repo
)
portfolio._positions = {}

sizer = FixedPercentageSizer(default_percentage=spec.execution.position_size_pct)
risk_manager = RiskManager(portfolio=portfolio, sizer=sizer)

# Define delayed execution and slippage models from spec
delay_model = KBarExecuteDelay(k=spec.execution.execution_delay_k)
slippage_model = FlatPriceSlip(slippage_pct=spec.execution.slippage_pct)

execution_handler = SimulatedExecutionHandler(
    delay_model=delay_model,
    slippage_model=slippage_model,
    execution_price_source="close",
    initial_balances={spec.execution.quote_currency: spec.execution.initial_balance}
)

execution_engine = ExecutionEngine(
    execution_handler=execution_handler,
    portfolio=portfolio,
    run_id=run_id
)

pipeline = TradingPipeline(
    ingestion=None,
    strategy=strategy_engine,
    risk=risk_manager,
    execution=execution_engine,
    portfolio=portfolio
)

prediction_logger = DatabasePredictionLogger(
    db=db,
    commit=False,
    model_class=BacktestPredictionLog,
    run_id=run_id
)
pipeline.prediction_logger = prediction_logger

2026-07-03 14:34:15,630 - INFO - Portfolio initialized with cash: 10000.00 USD
2026-07-03 14:34:15,633 - INFO - FixedPercentageSizer initialized with percentage: 10.00%
2026-07-03 14:34:15,634 - INFO - RiskManager initialized with sizer: fixed_percentage, max_allocation: 25.0%, max_positions: 10
2026-07-03 14:34:15,634 - INFO - ExecutionEngine initialized with handler for: simulated, max_retries=3, auto_reconciliation=True


### 5. Run Replay Loop & Record Portfolio Equity

We initialize the data reader to stream BTC/USDT bars between June 1st, 2026, and June 21st, 2026. During the execution of the replay loop, we record the portfolio's cash, position value, and total equity at each tick to construct our equity curve.

In [ ]:
data_reader = SQLBacktestDataReader(
    session=db,
    instrument_id=spec.market.instrument_id,
    start_date=spec.dates.start_date,
    end_date=spec.dates.end_date,
    warmup_bars=spec.warmup_bars,
    lookback_limit=spec.lookback_limit
)
loop_driver = HistoricalReplayLoop(data_reader=data_reader)

print("Starting simulation loop...")
from trading_bot.backtesting.engine import BacktestEngine

# Initialize and Run Backtest Engine
backtest_engine = BacktestEngine(
    pipeline=pipeline,
    data_reader=data_reader,
    db=db,
    instrument_id=spec.market.instrument_id
)

# Run simulation and clear previous DB entries matching run_id
result = backtest_engine.run(run_id=run_id, clear_previous_run=True)

# Extract and Save Performance Summary Stats
summary = result.to_dict()
os.makedirs(spec.output_dir, exist_ok=True)
result.save_summary(os.path.join(spec.output_dir, f"backtest_summary_{run_id}.json"))

print("--- BACKTEST SUMMARY STATS ---")
print(json.dumps(summary, indent=4))

2026-07-03 14:34:15,699 - INFO - Isolating backtest data by binding database session to: sqlite+pysqlite:////home/alfred/github/fts/backtests.db
2026-07-03 14:34:15,707 - INFO - Clearing previous database logs for run_id: presentation_lstm_ohlcv


Starting simulation loop...


2026-07-03 14:34:15,778 - INFO - Starting backtest engine simulation for run_id: presentation_lstm_ohlcv
2026-07-03 14:34:15,785 - INFO - Starting HistoricalReplayLoop simulation...
2026-07-03 14:34:15,844 - INFO - StrategyEngine generated a total of 1 signals this tick.
2026-07-03 14:34:15,855 - INFO - RiskManager approved order for BTC/USDT: buy 0.01 shares @ $73960.0000
2026-07-03 14:34:15,856 - INFO - Risk approved order for BTC/USDT (size: 0.0135 shares).
2026-07-03 14:34:15,860 - INFO - [nets_strategy_lstm] Executing order for BTC/USDT: buy 0.0135 shares @ $73960.0000
2026-07-03 14:34:15,863 - INFO - [SimulatedExecutionHandler] Queued order sim-060ec29f (buy) at tick 1, scheduled to fill at tick 2.
2026-07-03 14:34:15,865 - INFO - Handler returned result for BTC/USDT: ID: sim-060ec29f, Status: OrderStatus.OPEN
2026-07-03 14:34:15,898 - INFO - [SimulatedExecutionHandler] Filled order sim-060ec29f at price 73958.8850 (base price: 73885.0000, side: buy).
2026-07-03 14:34:15,905 - IN

--- BACKTEST SUMMARY STATS ---
{
    "run_id": "presentation_lstm_ohlcv",
    "instrument_id": "BTC/USDT",
    "strategy_name": "nets_strategy_lstm",
    "initial_equity": 10000.0,
    "final_equity": 9468.455023903536,
    "total_return_pct": -5.315449760964638,
    "max_drawdown_pct": -6.607243956274042,
    "sharpe_ratio": -9.184779917980299,
    "total_trades": 429
}


### 7. Render Interactive Dashboard

We load our interactive `BacktestVisualizer` pointing to the SQLite database and render the interactive dashboard to visually inspect cumulative returns, positions, and trades overlaying the candlestick chart.

In [ ]:
# Instantiate the visualizer pointing to the database
viz = BacktestVisualizer()

# Display the dashboard (incorporates ONNX model structure details if available)
viz.show_dashboard(onnx_model_path=onnx_path)

Output()

Output()

HTML(value="<hr style='border-color:#37474f;'/>")

HTML(value='<h3>🧬 ONNX Model Weight Inspector</h3>')

Output()

### 8. Export Standalone Interactive Visualization Report

Finally, we write the entire interactive visualization charts out to a standalone HTML file inside `runs/reports/` for offline review.

In [ ]:
exporter = HTMLBacktestExporter(visualizer=viz)
report_path = exporter.export(
    instrument_id=spec.market.instrument_id,
    strategy_name=strategy.name,
    run_id=run_id,
    output_path=spec.output_dir
)
print("Interactive HTML report successfully exported to:", report_path)
db.close()

2026-07-03 14:34:48,473 - INFO - Saving interactive backtest visualization HTML to: ../runs/reports/report_BTC_USDT_nets_strategy_lstm_presentation_lstm_ohlcv.html


Interactive HTML report successfully exported to: ../runs/reports/report_BTC_USDT_nets_strategy_lstm_presentation_lstm_ohlcv.html
